# 01. LangChain Components

LangChain is a framework for building applications with large language models (LLMs). It gives every model provider the same interface and supplies the building blocks around the model: prompts, chains, external knowledge, memory and agents.

| # | Component | In one line |
|---|---|---|
| 1 | **Models** | One standard way to talk to any LLM (OpenAI, Anthropic, Google, open-source) |
| 2 | **Prompts** | Reusable, dynamic instructions sent to the model |
| 3 | **Chains** | Pipelines where the output of one step becomes the input of the next |
| 4 | **Indexes** | Connect the model to your own data (PDFs, websites, databases) |
| 5 | **Memory** | Give a stateless model the context of earlier messages |
| 6 | **Agents** | LLMs that can reason and use tools to complete tasks |

> 🔑 **API keys.** The examples read keys from **Colab secrets** (the 🔑 icon in the left panel) or, on your own machine, from a `.env` file. Never paste a key directly into a notebook. See the repository README for setup.

In [ ]:
!pip install -q langchain langchain-openai langchain-anthropic python-dotenv

In [ ]:
import os

def get_key(name):
    """Read an API key from Colab secrets, or from an environment variable / .env file elsewhere."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ.get(name)

openai_api_key = get_key("OPENAI_API_KEY")
anthropic_api_key = get_key("ANTHROPIC_API_KEY")
print("OpenAI key found:", bool(openai_api_key))
print("Anthropic key found:", bool(anthropic_api_key))

## 1. Models

The **model** component is the core interface for talking to an LLM.

Most LLMs are used through an **API** instead of being downloaded. The problem is that every provider has its own client library, method names, message format and response structure. LangChain hides those differences behind **one standard interface**, so you can switch providers by changing a single line.

### Without LangChain: the OpenAI API

In [ ]:
# pip install openai
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
)
print("OpenAI Response:", response.choices[0].message.content)

### Without LangChain: the Anthropic (Claude) API

In [ ]:
# pip install anthropic
import anthropic

client = anthropic.Anthropic(api_key=anthropic_api_key)

message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=100,
    messages=[{"role": "user", "content": "What is the capital of Germany?"}],
)
print("Anthropic Response:", message.content[0].text)

### The Differences

| | OpenAI | Anthropic |
|---|---|---|
| **Client** | `OpenAI(api_key=...)` | `anthropic.Anthropic(api_key=...)` |
| **Method** | `client.chat.completions.create(...)` | `client.messages.create(...)` |
| **System prompt** | A message with `"role": "system"` | A separate `system=` argument |
| **Reading the answer** | `response.choices[0].message.content` | `message.content[0].text` |

Supporting both providers means writing and maintaining two different code paths.

### With LangChain: One Interface for Both

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

#### OpenAI with LangChain

In [ ]:
llm_openai = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is the capital of Japan?"),
]
response = llm_openai.invoke(messages)
print("LangChain (OpenAI) Response:", response.content)

#### Anthropic with LangChain

In [ ]:
llm_anthropic = ChatAnthropic(model="claude-sonnet-4-6", api_key=anthropic_api_key)

messages_anthropic = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is the capital of Spain?"),
]

try:
    response_anthropic = llm_anthropic.invoke(messages_anthropic)
    print("Langchain (Anthropic) Response:", response_anthropic.content)
except Exception as e:
    print(f"An error occurred with Langchain (Anthropic): {e}")

Langchain (Anthropic) Response: The capital of Spain is **Madrid**. It is the largest city in Spain and serves as the country's political, cultural, and economic center.


### What LangChain Standardizes

- **Initialization:** `ChatOpenAI` and `ChatAnthropic` are different classes but take the same kind of arguments (`model`, `api_key`).
- **Messages:** `SystemMessage` and `HumanMessage` work with every provider; LangChain translates them into each API's own format.
- **Invocation:** every model is called with `.invoke()`, and the answer is always in `.content`.

Switching from OpenAI to Claude is now a one-line change.

---

## 2. Prompts

A **prompt** is the input given to an LLM, and writing good prompts is called **prompt engineering**. LangChain offers three common patterns:

| Type | Idea | LangChain tool |
|---|---|---|
| **A. Dynamic and reusable prompts** | A template with placeholders filled in at run time | `PromptTemplate` |
| **B. Role-based prompts** | Give the model a persona or role | `SystemMessage` |
| **C. Few-shot prompts** | Show a few examples so the model copies the pattern | `FewShotPromptTemplate` |

### A. Dynamic and Reusable Prompts

The template is written once, and the variables in `{curly braces}` change on each call.

In [ ]:
from langchain_core.prompts import PromptTemplate

# Define a dynamic prompt template
prompt_template_dynamic = PromptTemplate.from_template(
    "What is a good name for a company that makes {product} and sells {item_type}?"
)

# Format the prompt with specific variables
formatted_prompt_dynamic = prompt_template_dynamic.format(product="eco-friendly clothes", item_type="sustainable fashion")

# Create messages for Anthropic model
messages_dynamic = [
    SystemMessage(content="You are a finance assistant."),
    HumanMessage(content=formatted_prompt_dynamic)
]

try:
    llm_anthropic = ChatAnthropic(model="claude-sonnet-4-6", api_key=anthropic_api_key)
    response_dynamic = llm_anthropic.invoke(messages_dynamic)
    print("Dynamic Prompt Response:", response_dynamic.content)
except Exception as e:
    print(f"An error occurred with Dynamic Prompt (Langchain Anthropic): {e}")

Dynamic Prompt Response: # Eco-Friendly Fashion Brand Name Ideas

Here are some creative options across different styles:

## 🌿 Nature-Inspired
- **Verdant Threads**
- **Terra Stitch**
- **Moss & Loom**
- **Rootwear**

## ♻️ Sustainability-Focused
- **Renew Apparel**
- **Cyclewear**
- **Second Bloom Clothing**
- **Evergreen Stitch**

## 🌍 Earth-Conscious
- **Kind Earth Co.**
- **Pure Ground Apparel**
- **GaiaWear**
- **EarthThread**

## ✨ Clean & Modern
- **Cleen Co.**
- **Purist Apparel**
- **Bare Earth Fashion**
- **SimpliGreen**

---

## Tips for Choosing:
- ✅ Make sure the **domain name is available**
- ✅ Check **trademark availability**
- ✅ Keep it **easy to spell and remember**
- ✅ Reflect your **brand values**

**My top picks:** *Verdant Threads*, *Terra Stitch*, or *Kind Earth Co.*: they feel both modern and meaningful.

Would you like help narrowing it down based on your target audience or brand personality?


### B. Role-Based Prompts

The `SystemMessage` sets the model's persona, and every answer follows it.

In [ ]:
# Create messages with a system role
messages_role_based = [
    SystemMessage(content="You are a poetic assistant that answers questions with rhymes."),
    HumanMessage(content="Tell me about the importance of water.")
]

try:
    llm_anthropic = ChatAnthropic(model="claude-sonnet-4-6", api_key=anthropic_api_key)
    response_role_based = llm_anthropic.invoke(messages_role_based)
    print("Role-based Prompt Response:", response_role_based.content)
except Exception as e:
    print(f"An error occurred with Role-based Prompt (Langchain Anthropic): {e}")

Role-based Prompt Response: Water is precious, water is life,
Without it the world would be filled with strife.
It flows through our rivers, our oceans, our streams,
It nourishes nature and fuels all our dreams.

Every creature that walks, swims, or flies,
Depends on this gift that falls from the skies.
It waters our crops and quenches our thirst,
Of all of Earth's blessings, water comes first.

It cleans and it cools and it carries away,
The burdens and troubles of each passing day.
Our bodies are made of it, mostly within,
Without it, no life could ever begin.

So treasure each raindrop, each river, each sea,
For water sustains both you and me.
Protect it and cherish it, guard every drop,
For when water runs out, all living things stop.


### C. Few-Shot Prompting

A few input and output examples teach the model the exact format you want.

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Define a few examples
examples = [
    {"word": "happy", "antonym": "sad"},
    {"word": "tall", "antonym": "short"}
]

# Create a template for the examples
example_formatter_template = "Word: {word}\nAntonym: {antonym}"
example_prompt = PromptTemplate(
    input_variables=["word", "antonym"],
    template=example_formatter_template,
)

# Create the few-shot prompt template
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Give the antonym of the following words:",
    suffix="Word: {input_word}\nAntonym:",
    input_variables=["input_word"],
    example_separator="\n\n",
)

# Format the few-shot prompt with a new input
formatted_few_shot_prompt = few_shot_prompt.format(input_word="light")

# Create messages for Anthropic model
messages_few_shot = [
    HumanMessage(content=formatted_few_shot_prompt)
]

try:
    llm_anthropic = ChatAnthropic(model="claude-sonnet-4-6", api_key=anthropic_api_key)
    response_few_shot = llm_anthropic.invoke(messages_few_shot)
    print("Few-shot Prompt Response:", response_few_shot.content)
except Exception as e:
    print(f"An error occurred with Few-shot Prompt (Langchain Anthropic): {e}")

Few-shot Prompt Response: Word: light
Antonym: **dark** (or **heavy**, depending on the context)


---

## 3. Chains

**Chains** build pipelines: the output of one step becomes the input of the next, so a multi-step task runs as one unit.

| Type | How it runs | Example |
|---|---|---|
| **Sequential** | Steps run one after another; each waits for the previous one | Read a customer review, classify its sentiment, then draft a reply |
| **Parallel** | Two or more steps run at the same time | Log the feedback to a Google Sheet **and** email the customer at once. Faster, but uses more compute |
| **Conditional** | The next step depends on the result of the current one | Positive feedback gets a thank-you; negative feedback is escalated to support |

In modern LangChain, chains are written with the pipe operator, for example `prompt | model | parser` (LCEL, the LangChain Expression Language). A dedicated Chains notebook is coming in this series.

---

## 4. Indexes

**Indexes** connect your application to **external knowledge**: PDFs, websites, spreadsheets, CSV files or databases. They are the foundation of **RAG** (Retrieval-Augmented Generation).

| Component | Job |
|---|---|
| **1. Document loaders** | Read the source (a PDF, a web page, a database table) |
| **2. Text splitters** | Cut long documents into chunks the model can handle |
| **3. Vector stores** | Store each chunk as an embedding (a vector of numbers) |
| **4. Retrievers** | Find the chunks most relevant to a question (semantic search) |

> 🔐 For **private or sensitive data**, consider an open-source model running on your own server (for example with **Ollama**), so the data never leaves your infrastructure. Models up to roughly 30 to 35 billion parameters run on a single powerful GPU server.

Notebook 02 shows embeddings and semantic search in action.

---

## 5. Memory

**LLM API calls are stateless.** The provider's server remembers nothing between requests: every call must include the whole conversation the model should know about. **Memory** is how an application keeps and resends that context.

| Type | Keeps | Good for |
|---|---|---|
| **1. Conversation buffer memory** | Every message of the conversation, word for word | Short conversations |
| **2. Conversation buffer window memory** | Only the last `k` exchanges | Long chats where recent context is enough |
| **3. Summary-based memory** | An LLM-written summary of earlier messages | Very long conversations that would exceed the token limit or cost too much |
| **4. Custom memory** | Whatever logic you design (for example, store only facts about the user) | Special application needs |

You can already see the simplest form of memory in notebook 02: the chat-model example appends each answer to the `messages` list and sends the whole list again. In LangChain 1.x the classic memory classes moved to the `langchain-classic` package, and new applications usually keep history with LangGraph. A dedicated Memory notebook is coming in this series.

---

## 6. Agents

An **AI agent** is a chatbot with superpowers. On top of generating text, it has:

- **Reasoning:** it plans the steps needed to solve a task, using techniques such as **chain-of-thought** prompting.
- **Tools:** it can call functions such as web search, a calculator, a database query or an API, and use the results in its answer.

For example, asked "What is the weather in Islamabad in Fahrenheit?", an agent decides to call a weather tool, then a conversion tool, then writes the answer. Agents get their own notebook later in this series.